# RAG Pipeline

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will build a Retrieval-Augmented Generation pipeline that grounds LLM answers in your own documents.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Create Documents

In production you'd use loaders. Here we create documents directly.

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(page_content="Python is a high-level programming language known for readability.", metadata={"source": "python-intro"}),
    Document(page_content="Machine learning is a subset of AI that learns patterns from data.", metadata={"source": "ml-intro"}),
    Document(page_content="FastAPI is a modern Python web framework for building APIs.", metadata={"source": "fastapi-intro"}),
    Document(page_content="Neural networks are computing systems inspired by biological brains.", metadata={"source": "nn-intro"}),
    Document(page_content="Docker containers package applications with their dependencies.", metadata={"source": "docker-intro"}),
]

print(f"Loaded {len(documents)} documents")

## 4. Text Splitting

Break large documents into smaller, overlapping chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)

long_doc = Document(
    page_content="Python was created by Guido van Rossum and released in 1991. "
    "It emphasizes code readability with significant whitespace. "
    "Python supports multiple programming paradigms including structured, "
    "object-oriented, and functional programming. "
    "It has a comprehensive standard library.",
    metadata={"source": "python-history"},
)

chunks = splitter.split_documents([long_doc])
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk.page_content[:80]}...")

## 5. Embeddings and Vector Store

Convert documents to vectors and index them for similarity search.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(documents)

print("Documents indexed.")

## 6. Similarity Search

In [ ]:
results = vector_store.similarity_search("What is machine learning?", k=2)
for doc in results:
    print(f"[{doc.metadata['source']}] {doc.page_content}")

## 7. The Retriever Pattern

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

docs = retriever.invoke("Tell me about Python web frameworks")
for doc in docs:
    print(f"[{doc.metadata['source']}] {doc.page_content}")

## 8. Build a 2-Step RAG Chain

Retrieve documents, format as context, and pass to the LLM.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

model = init_chat_model("gpt-4o-mini", model_provider="openai")

template = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based only on the following context:\n\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | template
    | model
)

response = rag_chain.invoke("What is Python used for?")
print(response.content)

## 9. Agentic RAG

Give an agent a retriever tool so it decides when and how to search.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for information about technology topics."""
    docs = retriever.invoke(query)
    return "\n\n".join(f"[{doc.metadata['source']}] {doc.page_content}" for doc in docs)

agent = create_react_agent(
    model=model,
    tools=[search_knowledge_base],
    prompt="You are a technical assistant. Use the knowledge base to find information before answering.",
)

result = agent.invoke({
    "messages": [HumanMessage(content="Compare Python and Docker — what does each do?")]
})
print(result["messages"][-1].content)

## Summary

- RAG grounds LLM responses in your own data
- `RecursiveCharacterTextSplitter` breaks documents into overlapping chunks
- Embeddings convert text to vectors for similarity search
- Vector stores index documents; `.as_retriever()` provides a standard interface
- 2-step RAG chains follow a fixed retrieve → generate flow
- Agentic RAG wraps the retriever as a tool for dynamic search control